# Notebook limpio: ProtT3-Gemma sobre Protein2Text-QA
Flujo general del experimento: configuración, datos, embeddings, modelo multimodal, entrenamiento y evaluación.

In [24]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tqdm import tqdm
import os
import hashlib
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    AutoConfig,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,)

In [25]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ESM_ID = "facebook/esm2_t30_150M_UR50D"
QFORMER_ID = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
GEMMA_ID = "google/gemma-2-2b-it"
CACHE_DIR = "./esm_cache"
N_QUERY_TOKENS = 32
MAX_PROT_LEN = 1024
MAX_TEXT_LEN = 768

os.makedirs(CACHE_DIR, exist_ok=True)

# Configuración del experimento
Se fijan rutas, modelos base, tamaños de secuencia, hiperparámetros y parámetros de entrenamiento.

# Carga y muestreo de datos
Se descarga Protein2Text-QA, se selecciona una muestra del split de prueba y se divide en entrenamiento y validación.

In [ ]:
HF_TOKEN = 'AQUI_EL_TOKEN'

login(token=HF_TOKEN)

In [27]:
p2tqa = load_dataset("tumorailab/Protein2Text-QA")

min_samp  = len(p2tqa["test"])*0.30
min_samp = int(min_samp)

test_p2tqa = p2tqa["test"]

data_small = test_p2tqa.shuffle(seed=42).select(range(min_samp))
split = data_small.train_test_split(test_size=0.05, seed=42)

train_data = split["train"]
val_data = split["test"]

train_data_df=train_data.to_pandas()
val_data_df=val_data.to_pandas()

train_data_df = train_data_df.reset_index(drop=True)
val_data_df = val_data_df.reset_index(drop=True)

print(train_data_df.info())
train_data_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10996 entries, 0 to 10995
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   long_format_id  10996 non-null  object
 1   id              10996 non-null  object
 2   protein         10996 non-null  object
 3   pmc_id          10996 non-null  object
 4   amino_seq       10996 non-null  object
 5   conversations   10996 non-null  object
dtypes: object(6)
memory usage: 515.6+ KB
None


,long_format_id,id,protein,pmc_id,amino_seq,conversations
0,Q8N729_61,Q8N729,Neuropeptide W,"[PMC9739957, PMC6015710, PMC7659951, PMC637823...",MAWRPGERGAPASRPRLALLLLLLLLPLPSGAWYKHVASPRYHTVG...,"[{'from': 'human', 'value': '<protein_sequence..."
1,P30291_43,P30291,Wee1-like protein kinase,"[PMC7170294, PMC9827073, PMC8832878, PMC946515...",MSFLSRQQPPPPRRAGAACTLRQKLIFSPCSDCEEEEEEEEEEGSG...,"[{'from': 'human', 'value': '<protein_sequence..."
2,P10746_6,P10746,Uroporphyrinogen-III synthase,"[PMC10823327, PMC10988712, PMC8620571, PMC1081...",MKVLLLKDAKEDDCGQDPYIRELGLYGLEATLIPVLSFEFLSLPSF...,"[{'from': 'human', 'value': '<protein_sequence..."
3,P56537_16,P56537,Eukaryotic translation initiation factor 6,"[PMC11230244, PMC9405973, PMC11071164]",MAVRASFENNCEIGCFAKLTNTYCLVAIGGSENFYSVFEGELSDTI...,"[{'from': 'human', 'value': '<protein_sequence..."
4,A0A994J6R6_27,A0A994J6R6,Phosphotransferase,"[PMC11537053, PMC11559000, PMC11508181, PMC114...",MIAAQLLAYYFTELKDDQVKKIDKYLYAMRLSDETLIDIMTRFRKE...,"[{'from': 'human', 'value': '<protein_sequence..."


In [28]:
train_data_df.iloc[0]['conversations']

array([{'from': 'human', 'value': '<protein_sequence>\nIs the protein associated with any particular disease or condition?'},
       {'from': 'gpt', 'value': 'The protein is associated with anorexia nervosa.'}],
      dtype=object)

In [29]:
def extract_conversations(conversation):

    question = conversation[0]['value']
    question = question.replace('<protein_sequence>\n','')
    answer = conversation[1]['value']

    return question, answer

In [30]:
question,answer = extract_conversations(train_data_df.iloc[100]['conversations'])

print(question)
print(answer)

How does the acetylcholine released by stimulated brush cells affect nearby cells?
It excites nearby cells via muscarinic acetylcholine receptors.


# Preparación de preguntas y respuestas
Se separan las conversaciones del dataset en columnas explícitas de pregunta y respuesta.

In [31]:
for i in range(len(train_data_df)):
    question,answer = extract_conversations(train_data_df.iloc[i]['conversations'])
    train_data_df.at[i, 'question'] = question
    train_data_df.at[i, 'answer'] = answer

for i in range(len(val_data_df)):
    question,answer = extract_conversations(val_data_df.iloc[i]['conversations'])
    val_data_df.at[i, 'question'] = question
    val_data_df.at[i, 'answer'] = answer

In [32]:
train_data_df.iloc[0]['conversations']

array([{'from': 'human', 'value': '<protein_sequence>\nIs the protein associated with any particular disease or condition?'},
       {'from': 'gpt', 'value': 'The protein is associated with anorexia nervosa.'}],
      dtype=object)

In [33]:
print(train_data_df.iloc[0]['question'])
print(train_data_df.iloc[0]['answer'])

Is the protein associated with any particular disease or condition?
The protein is associated with anorexia nervosa.


# Codificación de proteínas con ESM-2
Se prepara ESM-2 como encoder proteico congelado y se cachean embeddings para no recalcularlos en cada época.

In [34]:
SEQ_COL =  'amino_seq'
QUESTION_COL = 'question'
ANSWER_COL = 'answer'

esm_tokenizer = AutoTokenizer.from_pretrained(ESM_ID)
esm_model = AutoModel.from_pretrained(ESM_ID).to(DEVICE)
esm_model.eval()

for p in esm_model.parameters():
    p.requires_grad = False


def seq_hash(seq):
    return hashlib.sha1(seq.encode("utf-8")).hexdigest()


def cache_path(seq):
    return os.path.join(CACHE_DIR, f"{seq_hash(seq)}.pt")


@torch.no_grad()
@torch.no_grad()
def precompute_esm_cache(df, seq_col=SEQ_COL, batch_size=4):
    pending_seqs = []

    def flush(batch_seqs):
        if not batch_seqs:
            return

        toks = esm_tokenizer(
            batch_seqs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROT_LEN,
        ).to(DEVICE)

        out = esm_model(**toks).last_hidden_state
        lengths = toks["attention_mask"].sum(dim=1).cpu().tolist()

        for i, seq in enumerate(batch_seqs):
            emb = out[i, :lengths[i]].detach().cpu().to(torch.float16)
            torch.save(emb, cache_path(seq))

    seqs = (
        df[seq_col]
        .dropna()
        .astype(str)
        .str.strip()
        .drop_duplicates()
        .tolist()
    )

    for seq in tqdm(seqs, desc="Precomputando ESM cache"):
        if not os.path.exists(cache_path(seq)):
            pending_seqs.append(seq)

        if len(pending_seqs) >= batch_size:
            flush(pending_seqs)
            pending_seqs = []

    flush(pending_seqs)

precompute_esm_cache(train_data_df, seq_col=SEQ_COL, batch_size=4)
precompute_esm_cache(val_data_df, seq_col=SEQ_COL, batch_size=4)

/home/fabianh/proyecto_pln/.venv_proyecto_pln/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t30_150M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Precomputando ESM cache: 100%|██████████| 413/413 [00:00<00:00, 157663.38it/s]


# Dataset y DataLoader
Se construye el dataset de PyTorch y el `collate_fn` que empaqueta embeddings proteicos, preguntas y respuestas.

In [35]:
from torch.utils.data import Dataset
import torch

class ProteinQADataset(Dataset):
    def __init__(self, df):
        self.data = df.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        seq = str(row[SEQ_COL]).strip()
        question = str(row[QUESTION_COL]).strip()
        answer = str(row[ANSWER_COL]).strip()

        protein_emb = torch.load(
            cache_path(seq),
            map_location="cpu"
        ).float()

        return {
            "protein_emb": protein_emb,
            "question": question,
            "answer": answer,
        }


def collate_fn(batch):
    protein_embs = [x["protein_emb"] for x in batch]
    questions = [x["question"] for x in batch]
    answers = [x["answer"] for x in batch]

    max_len = max(e.shape[0] for e in protein_embs)
    dim = protein_embs[0].shape[1]

    protein_batch = torch.zeros(len(batch), max_len, dim)
    protein_mask = torch.zeros(len(batch), max_len, dtype=torch.long)

    for i, emb in enumerate(protein_embs):
        L = emb.shape[0]
        protein_batch[i, :L] = emb
        protein_mask[i, :L] = 1

    return {
        "protein_emb": protein_batch,
        "protein_mask": protein_mask,
        "questions": questions,
        "answers": answers,
    }


train_loader = DataLoader(
    ProteinQADataset(train_data_df),
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    ProteinQADataset(val_data_df),
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn,
)

In [36]:
print(len(train_loader))
print(len(val_loader))

2749
290


# Q-Former
Se define el puente cross-modal que comprime los embeddings de ESM-2 en query tokens compatibles con el LLM.

In [37]:
class ProteinQFormer(nn.Module):
    def __init__(self, qformer_id, protein_dim, llm_hidden_size, num_query_tokens=32):
        super().__init__()

        q_config = AutoConfig.from_pretrained(qformer_id)
        q_config.is_decoder = True
        q_config.add_cross_attention = True
        q_config.cross_attention_hidden_size = q_config.hidden_size

        self.qformer = AutoModel.from_pretrained(
            qformer_id,
            config=q_config,
            ignore_mismatched_sizes=True,
        )

        self.query_tokens = nn.Parameter(
            torch.randn(1, num_query_tokens, q_config.hidden_size) * 0.02
        )

        if protein_dim != q_config.hidden_size:
            self.protein_proj = nn.Linear(protein_dim, q_config.hidden_size)
        else:
            self.protein_proj = nn.Identity()

        self.proj_to_llm = nn.Linear(q_config.hidden_size, llm_hidden_size)

    def forward(self, protein_emb, protein_mask):
        B = protein_emb.shape[0]

        query_tokens = self.query_tokens.expand(B, -1, -1)
        query_mask = torch.ones(
            B,
            query_tokens.shape[1],
            dtype=torch.long,
            device=protein_emb.device,
        )

        projected_protein_emb = self.protein_proj(protein_emb)

        out = self.qformer(
            inputs_embeds=query_tokens,
            attention_mask=query_mask,
            encoder_hidden_states=projected_protein_emb,
            encoder_attention_mask=protein_mask,
        )

        protein_tokens = self.proj_to_llm(out.last_hidden_state)

        return protein_tokens

# Modelo ProtT3-Gemma para QA
Se integra el Q-Former con Gemma cuantizado y adaptado con LoRA para generar respuestas condicionadas por proteína y pregunta.

In [38]:
class ProtT3GemmaQA(nn.Module):
    def __init__(self, protein_dim):
        super().__init__()

        self.tokenizer = AutoTokenizer.from_pretrained(GEMMA_ID)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.tokenizer.padding_side = "right"

        if torch.cuda.is_available():
            major, _ = torch.cuda.get_device_capability()
            compute_dtype = torch.bfloat16 if major >= 8 else torch.float16
        else:
            compute_dtype = torch.float32

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
        )

        self.llm = AutoModelForCausalLM.from_pretrained(
            GEMMA_ID,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=compute_dtype,
        )

        self.llm.config.use_cache = False

        self.llm = prepare_model_for_kbit_training(self.llm)

        lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            target_modules=["q_proj", "v_proj"],
            task_type=TaskType.CAUSAL_LM,
        )

        self.llm = get_peft_model(self.llm, lora_config)

        self.llm.config.use_cache = False

        if hasattr(self.llm, "generation_config"):
            self.llm.generation_config.use_cache = False

            if hasattr(self.llm.generation_config, "cache_implementation"):
                self.llm.generation_config.cache_implementation = None

        llm_hidden = self.llm.config.hidden_size

        self.protein_qformer = ProteinQFormer(
            qformer_id=QFORMER_ID,
            protein_dim=protein_dim,
            llm_hidden_size=llm_hidden,
            num_query_tokens=N_QUERY_TOKENS,
        )

        self.protein_qformer.to(self.llm_device())

    def llm_device(self):
        return self.llm.get_input_embeddings().weight.device

    def build_prompt(self, question):
        return (
            "<bos><start_of_turn>user\n"
            "You are given learned protein tokens extracted from an amino-acid sequence.\n"
            f"Question: {question}\n"
            "<end_of_turn>\n"
            "<start_of_turn>model\n"
        )

    def encode_text_batch(self, questions, answers):
        pad_id = self.tokenizer.pad_token_id

        input_ids_list = []
        labels_list = []

        for q, a in zip(questions, answers):
            prompt = self.build_prompt(q)
            answer = str(a) + self.tokenizer.eos_token

            prompt_ids = self.tokenizer(
                prompt,
                add_special_tokens=False,
            ).input_ids

            answer_ids = self.tokenizer(
                answer,
                add_special_tokens=False,
            ).input_ids

            if len(prompt_ids) + len(answer_ids) > MAX_TEXT_LEN:
                remaining = max(1, MAX_TEXT_LEN - len(prompt_ids))
                answer_ids = answer_ids[:remaining]

            ids = prompt_ids + answer_ids
            labels = [-100] * len(prompt_ids) + answer_ids

            input_ids_list.append(ids)
            labels_list.append(labels)

        max_len = max(len(x) for x in input_ids_list)

        input_ids = torch.full(
            (len(input_ids_list), max_len),
            pad_id,
            dtype=torch.long,
        )

        attention_mask = torch.zeros(
            (len(input_ids_list), max_len),
            dtype=torch.long,
        )

        labels = torch.full(
            (len(input_ids_list), max_len),
            -100,
            dtype=torch.long,
        )

        for i, ids in enumerate(input_ids_list):
            L = len(ids)

            input_ids[i, :L] = torch.tensor(ids, dtype=torch.long)
            attention_mask[i, :L] = 1
            labels[i, :L] = torch.tensor(labels_list[i], dtype=torch.long)

        return input_ids, attention_mask, labels

    def forward(self, protein_emb, protein_mask, questions, answers):
        device = self.llm_device()

        protein_emb = protein_emb.to(device)
        protein_mask = protein_mask.to(device)

        protein_tokens = self.protein_qformer(
            protein_emb.float(),
            protein_mask,
        )

        input_ids, text_mask, labels = self.encode_text_batch(
            questions,
            answers,
        )

        input_ids = input_ids.to(device)
        text_mask = text_mask.to(device)
        labels = labels.to(device)

        text_embeds = self.llm.get_input_embeddings()(input_ids)

        protein_tokens = protein_tokens.to(
            device=device,
            dtype=text_embeds.dtype,
        )

        inputs_embeds = torch.cat(
            [protein_tokens, text_embeds],
            dim=1,
        )

        protein_token_mask = torch.ones(
            protein_tokens.shape[:2],
            dtype=torch.long,
            device=device,
        )

        attention_mask = torch.cat(
            [protein_token_mask, text_mask],
            dim=1,
        )

        protein_labels = torch.full(
            protein_tokens.shape[:2],
            -100,
            dtype=torch.long,
            device=device,
        )

        labels = torch.cat(
            [protein_labels, labels],
            dim=1,
        )

        out = self.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            use_cache=False,
        )

        return out

    @torch.no_grad()
    def generate(self, protein_emb, protein_mask, question, max_new_tokens=128):
        """
        Generación greedy manual.

        Se evita self.llm.generate(...) porque Gemma2 + PEFT + inputs_embeds
        puede fallar por HybridCache en algunas versiones de transformers.
        """
        self.eval()

        device = self.llm_device()

        protein_emb = protein_emb.to(device)
        protein_mask = protein_mask.to(device)

        protein_tokens = self.protein_qformer(
            protein_emb.float(),
            protein_mask,
        )

        prompt = self.build_prompt(question)

        input_ids = self.tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
        ).input_ids.to(device)

        text_embeds = self.llm.get_input_embeddings()(input_ids)

        protein_tokens = protein_tokens.to(
            device=device,
            dtype=text_embeds.dtype,
        )

        inputs_embeds = torch.cat(
            [protein_tokens, text_embeds],
            dim=1,
        )

        attention_mask = torch.ones(
            inputs_embeds.shape[:2],
            dtype=torch.long,
            device=device,
        )

        generated_ids = []

        eos_id = self.tokenizer.eos_token_id
        pad_id = self.tokenizer.pad_token_id

        if pad_id is None:
            pad_id = eos_id

        for _ in range(max_new_tokens):
            outputs = self.llm(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                use_cache=False,
            )

            next_token_logits = outputs.logits[:, -1, :]
            next_token_id = torch.argmax(
                next_token_logits,
                dim=-1,
                keepdim=True,
            )

            token_id_int = next_token_id.item()

            if token_id_int == eos_id:
                break

            generated_ids.append(token_id_int)

            next_token_embeds = self.llm.get_input_embeddings()(next_token_id)

            inputs_embeds = torch.cat(
                [inputs_embeds, next_token_embeds],
                dim=1,
            )

            next_attention = torch.ones(
                (attention_mask.shape[0], 1),
                dtype=torch.long,
                device=device,
            )

            attention_mask = torch.cat(
                [attention_mask, next_attention],
                dim=1,
            )

        if len(generated_ids) == 0:
            return ""

        return self.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        )

In [39]:
sample_batch = next(iter(train_loader))
protein_dim = sample_batch["protein_emb"].shape[-1]

model = ProtT3GemmaQA(protein_dim=protein_dim)
model.protein_qformer.to(model.llm_device())

model.llm.print_trainable_parameters()

/tmp/ipykernel_473773/977415435.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  protein_emb = torch.load(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of BertModel were not initialized from the model checkpoint at microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.1.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.1.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.1.crossattention.output.dense.bias', 'bert.encoder.layer.1.crossattention.output.dense.weight', 'bert.encoder.layer.1.

trainable params: 1,597,440 || all params: 2,615,939,328 || trainable%: 0.0611


# Entrenamiento
Se configuran optimizador, scheduler y ciclo de entrenamiento con acumulación de gradiente.

In [40]:
from torch.optim import AdamW

qformer_params = []
lora_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue

    if "protein_qformer" in name:
        qformer_params.append(param)
    else:
        lora_params.append(param)

optimizer = AdamW(
    [
        {"params": qformer_params, "lr": 1e-4},
        {"params": lora_params, "lr": 1e-5},
    ],
    weight_decay=0.01,
)

EPOCHS = 5
GRAD_ACCUM = 8
MAX_GRAD_NORM = 1.0

num_training_steps = EPOCHS * len(train_loader) // GRAD_ACCUM
warmup_steps = int(0.03 * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=num_training_steps,
)

# Inicialización del modelo
Se infiere la dimensión de los embeddings proteicos y se instancia la arquitectura completa.

In [41]:
model.train()

global_step = 0

for epoch in range(EPOCHS):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    running_loss = 0.0

    for step, batch in enumerate(pbar):
        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=torch.cuda.is_available(),
        ):
            out = model(
                protein_emb=batch["protein_emb"],
                protein_mask=batch["protein_mask"],
                questions=batch["questions"],
                answers=batch["answers"],
            )

            loss = out.loss / GRAD_ACCUM

        loss.backward()
        running_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                MAX_GRAD_NORM,
            )

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            global_step += 1

        pbar.set_postfix({
            "loss": running_loss / (step + 1),
            "lr_lora": scheduler.get_last_lr()[-1],
        })

Epoch 1/5:   0%|          | 0/2749 [00:00<?, ?it/s]/tmp/ipykernel_473773/977415435.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  protein_emb = torch.load(
/home/fabia

In [42]:
SAVE_DIR = "./prott3_gemma_qa"

os.makedirs(SAVE_DIR, exist_ok=True)

model.llm.save_pretrained(os.path.join(SAVE_DIR, "gemma_lora"))
model.tokenizer.save_pretrained(os.path.join(SAVE_DIR, "tokenizer"))

torch.save(
    model.protein_qformer.state_dict(),
    os.path.join(SAVE_DIR, "protein_qformer.pt"),
)

In [44]:
import pandas as pd
import torch
from tqdm import tqdm

model.eval()

predictions = []
ground_truths = []
questions = []

for _, row in tqdm(val_data_df.iterrows(), total=len(val_data_df), desc="Generando predicciones"):
    question = str(row["question"]).strip()
    answer   = str(row["answer"]).strip()

    protein_emb = torch.load(cache_path(row[SEQ_COL])).unsqueeze(0).float()
    protein_mask = torch.ones(1, protein_emb.shape[1], dtype=torch.long)

    pred_text = model.generate(protein_emb, protein_mask, question)

    predictions.append(pred_text)
    ground_truths.append(answer)
    questions.append(row["question"])

df_results = pd.DataFrame({
    "prediction": predictions,
    "ground_truth": ground_truths,
    "question": questions
})

print("¡Predicciones listas!")
df_results.head()

Generando predicciones:   0%|          | 0/579 [00:00<?, ?it/s]

/tmp/ipykernel_473773/3610510538.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  protein_emb = torch.load(cache_path(row[SEQ_COL])).unsqueeze(0).float()
Generando predi

¡Predicciones listas!


,prediction,ground_truth,question
0,"Yes, it has been shown to interact with the pr...","Yes, it has been linked to the ApoE ε4 isoform...",Are there any known interactions between this ...
1,"Yes, the protein is capable of inhibiting othe...","Yes, the protein can act as an inhibitor for o...",Is the protein capable of inhibiting other sub...
2,"It is involved in cell proliferation, differen...",This protein plays a role in cytoskeletal remo...,What is the primary role of this protein in ce...
3,"Yes, the protein is involved in the regulation...","It is part of the LTBP4-TGFβ1-MDSCs axis, whic...",Is the protein involved in the regulation of a...
4,The protein is upregulated when ALOX15B is sup...,"The protein is still present, but its activati...",What happens to the protein when ALOX15B is su...


In [45]:
!pip install -q evaluate rouge_score sacrebleu

import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

preds = df_results["prediction"].tolist()
refs_bleu = [[gt] for gt in df_results["ground_truth"].tolist()]
refs_rouge = df_results["ground_truth"].tolist()

bleu_output = bleu.compute(predictions=preds, references=refs_bleu)
rouge_output = rouge.compute(predictions=preds, references=refs_rouge)

print("--- RESULTADOS DE EVALUACIÓN ---")
print(f"BLEU Score: {bleu_output['bleu']:.4f}")
print(f"ROUGE-1: {rouge_output['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_output['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_output['rougeL']:.4f}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


--- RESULTADOS DE EVALUACIÓN ---
BLEU Score: 0.2185
ROUGE-1: 0.4163
ROUGE-2: 0.2476
ROUGE-L: 0.3924


# Guardado del modelo
Se guardan los componentes entrenables y la configuración necesaria para reutilizar el modelo.

In [46]:
!pip install -q bert-score

bertscore = evaluate.load("bertscore")

bertscore_output = bertscore.compute(
    predictions=preds,
    references=refs_rouge,
    lang="en",
    model_type="distilbert-base-uncased",
)

print(f"BERTScore Precision: {sum(bertscore_output['precision']) / len(bertscore_output['precision']):.4f}")
print(f"BERTScore Recall:    {sum(bertscore_output['recall'])    / len(bertscore_output['recall']):.4f}")
print(f"BERTScore F1:        {sum(bertscore_output['f1'])        / len(bertscore_output['f1']):.4f}")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/home/fabianh/proyecto_pln/.venv_proyecto_pln/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


BERTScore Precision: 0.8585
BERTScore Recall:    0.8432
BERTScore F1:        0.8496


In [47]:
for i in range(10):
    row=df_results.iloc[i]
    prediccion = row['prediction']
    verdad=row['ground_truth']
    pregunta=row['question']
    print(f'id{i}')
    print(f'Pregunta: {pregunta}')
    print('\n')
    print(f'Predcción:{prediccion}')
    print('\n')
    print(f'Verdad: {verdad}')
    print('------------------------------------------------------------------------------------')


id0
Pregunta: Are there any known interactions between this protein and other proteins or genes?


Predcción:Yes, it has been shown to interact with the protein p53.


Verdad: Yes, it has been linked to the ApoE ε4 isoform, which is also associated with Alzheimer Disease.
------------------------------------------------------------------------------------
id1
Pregunta: Is the protein capable of inhibiting other substances?


Predcción:Yes, the protein is capable of inhibiting other substances, such as the enzyme 5-lipoxygenase.


Verdad: Yes, the protein can act as an inhibitor for other substances.
------------------------------------------------------------------------------------
id2
Pregunta: What is the primary role of this protein in cell processes?


Predcción:It is involved in cell proliferation, differentiation, and apoptosis.


Verdad: This protein plays a role in cytoskeletal remodeling, which supports several cellular events, including cell division, movement, and migration

# Generación de predicciones
Se evalúa el modelo sobre validación y se almacenan predicciones junto con las respuestas esperadas.

# Métricas automáticas
Se calculan BLEU, ROUGE y BERTScore para comparar las respuestas generadas contra las referencias.